In [11]:
import importlib
from pathlib import Path
import utils_database

importlib.reload(utils_database)

from utils_database import *

In [12]:
df = pd.read_excel("24ACE-FH-EARLY_DHM_v105_20260424.xls", sheet_name="Data Handling Manual FH-EARLY")
variables = df[df["Référence"].notna()].copy()

variables = variables[["Id", "Référence", "Saisie", "Type", "Format", "Intitulé"]]

In [13]:
export_folder = Path("export-example")

inc = read_export_csv(export_folder / "INC_ANSI.csv")
ml = read_export_csv(export_folder / "ML_ANSI.csv")
tab_bs = read_export_csv(export_folder / "TAB_BS_ANSI.csv")
tab_fh = read_export_csv(export_folder / "TAB_FH_ANSI.csv")

export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh}

In [14]:
for var in ["SEX", "BRTHDAT", "AGE"]:
    print(var, "->", find_export_columns(var, inc.columns))

# ml columns
print("FH7ALS ->", find_export_columns("FH7ALS", ml.columns))

SEX -> ['SEX']
BRTHDAT -> ['BRTHDAT_D', 'BRTHDAT_M', 'BRTHDAT_Y', 'BRTHDAT']
AGE -> ['AGE', 'AGE_V']
FH7ALS -> ['FH7ALS_C1', 'FH7ALS_C2', 'FH7ALS_C3', 'FH7ALS_C4', 'FH7ALS_C5', 'FH7ALS_C6', 'FH7ALS_C7', 'FH7ALS_C8', 'FH7ALS_C9', 'FH7ALS_C10', 'FH7ALS_C11', 'FH7ALS_C12', 'FH7ALS_C13', 'FH7ALS_C14']


In [15]:
export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh,}

mapped = create_dictionary_export_map(variables, export_tables)
summary, mismatched = summarize_dictionary_export_map(mapped)

In [16]:
mapped["Type_english"] = mapped["Type"].apply(classify_variable_type)
mapped[["Référence", "Type", "Type_english", "Intitulé", "INC_export_cols", "ML_export_cols", "TAB_BS_export_cols", "TAB_FH_export_cols"]].head()

,Référence,Type,Type_english,Intitulé,INC_export_cols,ML_export_cols,TAB_BS_export_cols,TAB_FH_export_cols
2,PROJECTNAME,Label,label,FH-Early,[],[],[],[]
3,SEX,Radio buttons,categorical,Gender,[SEX],[],[],[]
4,BRTHDAT,Date,date,Date of birth,"[BRTHDAT_D, BRTHDAT_M, BRTHDAT_Y, BRTHDAT]",[],[],[]
5,AGE,Entier,numeric,Age,"[AGE, AGE_V]",[],[],[]
6,INVNAME,Texte,text,Investigator,[INVNAME],[],[],[]


In [18]:
mapped["code_labels"] = mapped["Format"].apply(parse_code_labels)
mapped[["Référence", "Type", "Format", "code_labels"]].head(20)

,Référence,Type,Format,code_labels
2,PROJECTNAME,Label,NaN,{}
3,SEX,Radio buttons,1 = Male \n2 = Female,"{1: 'Male', 2: 'Female'}"
4,BRTHDAT,Date,Précision = MM/YYYY\nValeurs par défaut = 01/0...,{}
5,AGE,Entier,Taille : N\nMin = 18\nMax = 120,{}
6,INVNAME,Texte,Taille : N,{}
7,SITEID,Texte,Taille : N,{}
8,SUBJECT_REF,Texte,Taille : N,{}
9,RFICDAT,Date,Précision = JJ/MM/YYYY\nValeurs par défaut = 0...,{}
12,INF1YN,Radio buttons,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"
13,INF1AYN,Radio buttons,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"


In [ ]:
# check that code labels are captured for all categorical and checkbox variables
mapped[
    (mapped["Type_english"].isin(["categorical", "checkbox"])) &
    (mapped["code_labels"].apply(len) == 0)
][["Référence", "Type", "Format", "Intitulé"]]

,Référence,Type,Format,Intitulé


In [ ]:
# mapped.to_csv("dictionary_export_map.csv", index=False)

In [20]:
mapped[
    ["Référence", "Type", "Type_english", "Intitulé",
     "code_labels", "has_export_match", "n_export_matches"]
].head(20)

,Référence,Type,Type_english,Intitulé,code_labels,has_export_match,n_export_matches
2,PROJECTNAME,Label,label,FH-Early,{},False,0
3,SEX,Radio buttons,categorical,Gender,"{1: 'Male', 2: 'Female'}",True,1
4,BRTHDAT,Date,date,Date of birth,{},True,4
5,AGE,Entier,numeric,Age,{},True,2
6,INVNAME,Texte,text,Investigator,{},True,1
7,SITEID,Texte,text,Centre,{},True,1
8,SUBJECT_REF,Texte,text,Patient ID,{},True,5
9,RFICDAT,Date,date,Date of inclusion,{},True,4
12,INF1YN,Radio buttons,categorical,Patient deceased at the time of inclusion,"{1: 'Yes', 0: 'No'}",True,1
13,INF1AYN,Radio buttons,categorical,"If so, did you verify that there was no object...","{1: 'Yes', 0: 'No'}",True,1
